<a href="https://colab.research.google.com/github/Assaoka/Decolar--Introducao_a_Ciencia_de_Dados/blob/main/%5B%20%207%20%5D%20Limpeza%20e%20Agrupamento%20de%20Dados%20/%20Limpeza%20e%20Agrupamento%20de%20Dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 7: A Arte da Limpeza e Agrupamento de Dados


Na última aula, aprendemos a fazer resumos estatísticos, contagens e filtros avançados para fazer perguntas complexas à nossa Pokédex.

Hoje, vamos nos aprofundar em uma das tarefas mais importantes de um cientista de dados: a **limpeza e preparação dos dados**. Vamos aprender a lidar com informações faltantes, a agrupar dados para criar resumos poderosos e a transformar texto em números, um passo essencial para futuros modelos de Inteligência Artificial.


## 1. Carregando a Pokédex

Como sempre, nosso primeiro passo é importar o `pandas` e carregar nossos dados.


In [1]:
import pandas as pd

endereco = 'https://raw.githubusercontent.com/Assaoka/Decolar--Introducao_a_Ciencia_de_Dados/refs/heads/main/pokemon.csv'
df = pd.read_csv(endereco)
df.head()

,Name,National Dex #,Primary Typing,Secondary Typing,Secondary Typing Flag,Generation,Legendary Status,Form,Alt Form Flag,Evolution Stage,...,Weight (hg),Height (in),Weight (lbs),Base Stat Total,Health,Attack,Defense,Special Attack,Special Defense,Speed
0,bulbasaur,1,grass,poison,True,generation-i,False,Base,False,1,...,69,28,15,318,45,49,49,65,65,45
1,ivysaur,2,grass,poison,True,generation-i,False,Base,False,2,...,130,39,29,405,60,62,63,80,80,60
2,venusaur,3,grass,poison,True,generation-i,False,Base,False,3,...,1000,79,220,525,80,82,83,100,100,80
3,venusaur-mega,3,grass,poison,True,generation-i,True,Mega,True,3,...,1555,94,343,625,80,100,123,122,120,80
4,charmander,4,fire,NaN,False,generation-i,False,Base,False,1,...,85,24,19,309,39,52,43,60,50,65


## 2. Revisão Rápida: Poderes da Aula 6

Na última aula, você aprendeu a:

1. **`.describe()`**: Gerar um resumo estatístico completo (média, desvio padrão, mínimo, máximo, etc.) para colunas numéricas.



In [3]:
df[['Attack', 'Defense', 'Speed']].describe().T

,count,mean,std,min,25%,50%,75%,max
Attack,1184.0,80.989020,31.955337,5.0,57.0,80.0,100.0,190.0
Defense,1184.0,74.817568,30.324208,5.0,52.0,70.0,91.0,230.0
Speed,1184.0,69.728885,30.110391,5.0,45.0,67.5,91.0,200.0


2. **`.value_counts()`**: Contar a frequência de cada categoria em uma coluna.


In [4]:
df['Primary Typing'].value_counts()

,count
Primary Typing,
water,145
normal,128
grass,113
bug,89
fire,77
psychic,77
rock,71
electric,71
dark,57


3. **Filtros Compostos**: Usar `&` (E) e `|` (OU) para criar consultas complexas, como `df[(df['Primary Typing'] == 'grass') & (df['Legendary Status'] == True)]`.
    


In [6]:
df[(df['Primary Typing'] == 'grass') & (df['Legendary Status'] == True)].head()

,Name,National Dex #,Primary Typing,Secondary Typing,Secondary Typing Flag,Generation,Legendary Status,Form,Alt Form Flag,Evolution Stage,...,Weight (hg),Height (in),Weight (lbs),Base Stat Total,Health,Attack,Defense,Special Attack,Special Defense,Speed
3,venusaur-mega,3,grass,poison,True,generation-i,True,Mega,True,3,...,1555,94,343,625,80,100,123,122,120,80
208,meganium,154,grass,NaN,False,generation-ii,True,Base,False,3,...,1005,71,222,525,80,82,100,83,100,80
321,sceptile-mega,254,grass,dragon,True,generation-iii,True,Mega,True,3,...,552,75,122,630,70,110,75,145,85,145
559,abomasnow-mega,460,grass,ice,True,generation-iv,True,Mega,True,2,...,1850,106,408,594,90,132,105,132,105,30
597,shaymin-land,492,grass,NaN,False,generation-iv,True,Base,False,1,...,21,8,5,600,100,100,100,100,100,100


4. **`.map()` e `.apply()`**: Transformar colunas, aplicando dicionários ou funções customizadas para criar novas informações.


In [21]:
def categoria_peso(row):
    if row['Weight (hg)'] > 1000:
        return 'Pesado'
    else:
        return 'Leve'

df['Peso'] = df.apply(categoria_peso, axis=1)
df.sample(5)

,Name,National Dex #,Primary Typing,Secondary Typing,Secondary Typing Flag,Generation,Legendary Status,Form,Alt Form Flag,Evolution Stage,...,Height (in),Weight (lbs),Base Stat Total,Health,Attack,Defense,Special Attack,Special Defense,Speed,Peso
884,toxapex,748,poison,water,True,generation-vii,False,Base,False,2,...,28,32,495,50,63,152,53,142,35,Leve
399,wailord,321,water,NaN,False,generation-iii,False,Base,False,2,...,571,877,500,170,90,45,90,45,60,Pesado
187,omastar,139,rock,water,True,generation-i,False,Base,False,2,...,39,77,495,70,60,125,115,70,55,Leve
58,gloom,44,grass,poison,True,generation-i,False,Base,False,2,...,31,19,395,60,65,70,85,75,40,Leve
811,binacle,688,rock,water,True,generation-vi,False,Base,False,1,...,20,68,306,42,52,67,39,56,50,Leve


## 3. Tratando Dados Faltantes (`.fillna()`)

Ao explorar dados, é muito comum encontrar "buracos" ou valores vazios (chamados de `NaN` - _Not a Number_). Modelos de análise e machine learning não sabem como lidar com eles, então precisamos tratá-los.

Primeiro, vamos ver quantos valores faltantes temos em cada coluna:

In [22]:
# O comando .isnull() retorna True para cada célula que é NaN.
# O .sum() soma esses Trues (considerando True=1 e False=0)
df.isnull().sum()

,0
Name,0
National Dex #,0
Primary Typing,0
Secondary Typing,530
Secondary Typing Flag,0
Generation,0
Legendary Status,0
Form,0
Alt Form Flag,0
Evolution Stage,0


Vemos que a coluna `Secondary Typing` tem muitos valores faltando. Isso faz sentido, já que muitos Pokémon têm apenas um tipo. Em vez de deixar como `NaN`, vamos preencher esses campos com o texto "Nenhum". Para isso, usamos o método `.fillna()`.

In [27]:
# O método .fillna() preenche os valores NaN com o valor que especificarmos.
# O `inplace=True` modifica o DataFrame df diretamente, sem precisar fazer df = df.fillna(...)
df2 = df.copy()
df2['Secondary Typing'] = df['Secondary Typing'].fillna('Nenhum')
df2.isnull().sum()

,0
Name,0
National Dex #,0
Primary Typing,0
Secondary Typing,0
Secondary Typing Flag,0
Generation,0
Legendary Status,0
Form,0
Alt Form Flag,0
Evolution Stage,0


## 4. Agrupando Dados para Análise (`.groupby()`)

E se quisermos saber qual o tipo de Pokémon (`Primary Typing`) tem, em média, o maior ataque? Ou a maior defesa? Fazer isso com filtros seria muito trabalhoso.

Para isso, usamos o `.groupby()`, um dos comandos mais poderosos do Pandas. Ele segue uma lógica de **"Separar-Aplicar-Combinar"**:

1. **Separar**: Separa o DataFrame em grupos menores com base em uma ou mais colunas.
    
2. **Aplicar**: Aplica uma função a cada grupo para agregar os dados.
    
3. **Combinar**: Junta os resultados em um novo DataFrame.
    

### Funções de Agregação Comuns

Após o `.groupby()`, você pode aplicar diversas funções, como:

- `.mean()`: Calcula a média.
    
- `.sum()`: Soma os valores.
    
- `.min()`: Retorna o menor valor.
    
- `.max()`: Retorna o maior valor.
    
- `.count()`: Conta os valores não nulos.
    
- `.size()`: Conta o total de itens (incluindo nulos).
    
- `.median()`: Calcula a mediana.
    
- `.std()`: Calcula o desvio padrão.
    

### Agrupando por uma coluna

Vamos agrupar por 'Primary Typing' e calcular a média dos stats de batalha.

In [28]:
# Agrupando por uma coluna e aplicando uma função
stats_por_tipo = df.groupby('Primary Typing')[['Attack', 'Defense', 'Speed', 'Base Stat Total']].mean()

# Ordenando pelo 'Base Stat Total' para ver os tipos mais fortes em média
stats_por_tipo.sort_values(by='Base Stat Total', ascending=False)

,Attack,Defense,Speed,Base Stat Total
Primary Typing,,,,
dragon,105.979167,81.479167,85.395833,528.875000
steel,91.232558,115.627907,56.744186,485.232558
psychic,74.740260,70.831169,77.675325,482.389610
fighting,104.960000,76.440000,76.120000,458.080000
fire,84.415584,69.844156,73.584416,456.571429
rock,91.112676,95.774648,62.535211,454.746479
dark,85.228070,70.964912,77.526316,449.070175
fairy,69.451613,72.258065,68.548387,448.290323
electric,72.042254,65.830986,87.352113,447.014085


### Agrupando por Múltiplas Colunas

Podemos ser ainda mais específicos. E se quisermos ver a contagem de Pokémon por Geração E se eles são lendários ou não? Basta passar uma lista de colunas para o `.groupby()`.

In [29]:
# Agrupando por múltiplas colunas
# O .size() é ótimo para contar a quantidade de registros em cada subgrupo
contagem_geracao_lendarios = df.groupby(['Generation', 'Legendary Status']).size()
contagem_geracao_lendarios

Generation       Legendary Status
generation-i     False               183
                 True                 23
generation-ii    False                99
                 True                 13
generation-iii   False               130
                 True                 32
generation-iv    False                99
                 True                 21
generation-ix    False                91
                 True                 35
generation-v     False               154
                 True                 17
generation-vi    False                71
                 True                 11
generation-vii   False                74
                 True                 29
generation-viii  False                85
                 True                 17
dtype: int64

### Aplicando Múltiplas Funções com `.agg()`

E se você quiser ver o `min`, o `max` e a `media` do `Attack` para cada tipo, tudo de uma vez? Para isso, usamos o método `.agg()` (de "aggregate"), passando uma lista de funções que queremos aplicar.

In [31]:
# Usando .agg() para aplicar múltiplas funções
stats_detalhados_por_tipo = df.groupby('Primary Typing')['Attack'].agg(['min', 'max', 'mean'])
stats_detalhados_por_tipo

,min,max,mean
Primary Typing,,,
bug,10,185,71.853933
dark,28,150,85.228070
dragon,50,180,105.979167
electric,30,123,72.042254
fairy,20,150,69.451613
fighting,35,145,104.960000
fire,30,160,84.415584
flying,30,115,79.888889
ghost,30,165,70.666667


## 5. One-Hot Encoding Avançado: Lidando com Múltiplas Categorias

No mundo real, é comum que um item pertença a várias categorias ao mesmo tempo. Um filme pode ser "Ação", "Aventura" e "Ficção Científica". No nosso caso, um Pokémon pode ser do tipo "grass" e "poison".

O `get_dummies` padrão não funciona bem para isso. Precisamos de uma abordagem mais inteligente.

### Passo 1: Unificar os Tipos em uma Única Coluna

Primeiro, vamos criar uma nova coluna que junta o tipo primário e o secundário, separados por uma vírgula.

In [34]:
# Juntando as duas colunas de tipo em uma só
def all_types(row):
    if pd.isna(row['Secondary Typing']) or row['Secondary Typing'] == '':
        return row['Primary Typing']
    else:
        return row['Primary Typing'] + ',' + row['Secondary Typing']

df['All_Types'] = df.apply(all_types, axis=1)
display(df[['Name', 'Primary Typing', 'Secondary Typing', 'All_Types']].head())

,Name,Primary Typing,Secondary Typing,All_Types
0,bulbasaur,grass,poison,"grass,poison"
1,ivysaur,grass,poison,"grass,poison"
2,venusaur,grass,poison,"grass,poison"
3,venusaur-mega,grass,poison,"grass,poison"
4,charmander,fire,NaN,fire


### Passo 2: Usar `.str.get_dummies()` para Separar as Categorias

Agora que temos uma única coluna com os tipos separados por vírgula, podemos usar uma função especial: `.str.get_dummies()`. Ela entende que precisa quebrar a string no separador e criar as colunas a partir daí.

In [35]:
# O argumento `sep=','` diz à função para quebrar a string toda vez que encontrar uma vírgula
tipos_dummies = df['All_Types'].str.get_dummies(sep=',')

# Vamos ver o resultado para os primeiros Pokémon.
# Repare como o Bulbasaur tem '1' nas colunas 'grass' e 'poison', enquanto o Charmander só tem em 'fire'.
tipos_dummies.head()

,bug,dark,dragon,electric,fairy,fighting,fire,flying,ghost,grass,ground,ice,normal,poison,psychic,rock,steel,water
0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
4,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


Essa técnica é extremamente poderosa e prepara os dados da forma correta para algoritmos de Machine Learning quando temos múltiplas categorias por item.


## 6. Hora dos Exercícios!

Vamos praticar os novos conceitos.

### Exercício 1: Contagem de Tipos Secundários

Antes da limpeza, quantos Pokémon **não tinham** um `Secondary Typing`? (Dica: use o DataFrame original ou reconte os `NaN` antes do `.fillna()`).

### Exercício 2: O Pokémon mais pesado de cada tipo

Use `.groupby()` para encontrar o Pokémon com o maior peso (`Weight (hg)`) para cada `Primary Typing`. (Dica: use `.max()` após o groupby).


### Exercício 3: Análise dos Lendários

Use `.groupby()` na coluna `Legendary Status` para comparar a média de `Attack`, `Defense` e `Speed` entre Pokémon lendários e não lendários.

### Exercício 4: Agrupando por duas colunas

Agrupe o DataFrame por `Generation` e `Legendary Status` para ver a contagem de Pokémon em cada categoria. (Dica: use `.size()` após o groupby para contar).


### Exercício 5: Juntando os Resultados

Para criar um DataFrame final pronto para Machine Learning, precisamos juntar nosso DataFrame original com as `tipos_dummies` que criamos. Use `pd.concat()` para isso.
